In [ ]:
#%pip install geopandas shapely

## 1. Загрузка словаря геометрий

Читаем CSV-словарь `ЦМУТ_Сервисы - сервисы_ЦМУТ.csv` и формируем множество имён сервисов,
для которых требуется геометрия типа **полигон**.  
Остальные сервисы сохраняются как точки (центроиды).

In [7]:
import pandas as pd
import geopandas as gpd
from shapely.validation import make_valid
from pathlib import Path
import warnings

# Путь к словарю (в той же папке, что и ноутбук)
DICT_PATH = Path("ЦМУТ_Сервисы - сервисы_ЦМУТ.csv")

# CSV: столбец 2 — имя файла (без .geojson), столбец 7 — тип геометрии
dict_df = pd.read_csv(DICT_PATH, header=None, encoding="utf-8-sig")

geometry_col = dict_df.iloc[:, 7].astype(str).str.strip().str.lower()
name_col = dict_df.iloc[:, 2].astype(str).str.strip()

polygon_services = set(name_col[geometry_col == "полигон"].tolist())
polygon_services.discard("nan")
polygon_services.discard("")

print(f"Сервисы с геометрией 'полигон' ({len(polygon_services)} шт.):")
print(sorted(polygon_services))

Сервисы с геометрией 'полигон' (36 шт.):
['Botanical_garden', 'Thermal_pp', 'agricultural_services', 'airport', 'beach', 'cemetery', 'chemical', 'construction', 'dog_park', 'electronics_industry', 'embankment', 'energy', 'engineering', 'extractive', 'farmland', 'food_industry', 'forest', 'golf', 'heat_pp', 'historic', 'hydroelectric_pp', 'landfill', 'light_industry', 'metallurgy', 'nuclear_pp', 'oopt', 'park', 'parking', 'port', 'reserve', 'skateboard', 'theme_park', 'wastewater_plant', 'water_park', 'woodworking', 'zoo']


## 2. Вспомогательные функции обработки

- **`safe_read`** — читает GeoJSON, удаляя невалидные/пустые геометрии
- **`fix_geometries`** — исправляет невалидные геометрии через `make_valid()`
- **`remove_intersecting_points`** — удаляет точки, перекрывающиеся полигонами
- **`convert_polygons_to_points`** — конвертирует полигоны в репрезентативные точки

In [8]:
def safe_read(filepath: str) -> gpd.GeoDataFrame:
    """
    Читает GeoJSON-файл с обработкой невалидных геометрий:
    1. on_invalid='ignore' — заменяет нечитаемые геометрии на None
    2. Удаляет строки с None-геометрией
    3. Исправляет оставшиеся невалидные геометрии через make_valid()
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gdf = gpd.read_file(filepath, on_invalid="ignore")

    # Удаляем строки с None или пустой геометрией
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    if gdf.empty:
        return gdf

    # Исправляем оставшиеся невалидные геометрии
    invalid_mask = ~gdf.geometry.is_valid
    if invalid_mask.any():
        count = invalid_mask.sum()
        print(f"    [fix] исправлено невалидных геометрий: {count}")
        gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].apply(make_valid)
        gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]

    return gdf


def remove_intersecting_points(points_gdf, polygons_gdf):
    """
    Удаляет точки, которые попадают внутрь полигонов или пересекают их.
    """
    if points_gdf.empty or polygons_gdf.empty:
        return points_gdf

    joined = points_gdf.sjoin(polygons_gdf, how="left", predicate="intersects")
    clean_points = joined[joined["index_right"].isna()].copy()

    cols_to_drop = [col for col in clean_points.columns if col.endswith("_right") or col == "index_right"]
    clean_points = clean_points.drop(columns=cols_to_drop)
    clean_points.columns = clean_points.columns.str.replace("_left$", "", regex=True)

    return clean_points


def convert_polygons_to_points(polygons_gdf):
    """
    Преобразует полигоны в точки, гарантируя, что точка лежит внутри полигона.
    Использует representative_point() вместо centroid() для надёжности.
    """
    if polygons_gdf.empty:
        return polygons_gdf

    points_from_polygons = polygons_gdf.copy()
    points_from_polygons["geometry"] = points_from_polygons.geometry.representative_point()
    return points_from_polygons

## 3. Главная функция обработки одного файла

Логика ветвления на основе словаря:
- **polygon** → оставить только полигоны, отбросить точки и линии
- **центроид** (остальные) → исправление геометрий, фильтрация пересечений и конвертация полигонов в точки

In [9]:
def process_service_file(input_path: Path, output_path: Path, polygon_services: set) -> str:
    """
    Обрабатывает один GeoJSON-файл:
    - если сервис в polygon_services → сохраняет только полигоны
    - иначе → конвейер: очистка точек + перевод полигонов в точки
    Возвращает строку-статус для итогового отчёта.
    """
    service_name = input_path.stem
    gdf = safe_read(str(input_path))

    if gdf.empty:
        return f"⚠️  {service_name}: файл пустой или все геометрии невалидны — пропущен"

    if service_name in polygon_services:
        # ── Ветка POLYGON: сохраняем только полигоны ──
        result_gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

        if result_gdf.empty:
            return f"⚠️  {service_name}: ожидались полигоны, но их нет — пропущен"

        dropped = len(gdf) - len(result_gdf)
        result_gdf.to_file(output_path, driver="GeoJSON")
        return (
            f"✅  {service_name}: сохранено {len(result_gdf)} полигонов"
            + (f" (отброшено {dropped} не-полигонов)" if dropped else "")
        )

    else:
        # ── Ветка CENTROID: стандартный конвейер ──
        points = gdf[gdf.geometry.type.isin(["Point", "MultiPoint"])].copy()
        polygons = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

        clean_points = remove_intersecting_points(points, polygons)
        polygons_as_points = convert_polygons_to_points(polygons)

        final_gdf = pd.concat([clean_points, polygons_as_points], ignore_index=True)

        if final_gdf.empty:
            return f"⚠️  {service_name}: результат пустой — файл не сохранён"

        final_gdf.to_file(output_path, driver="GeoJSON")
        return (
            f"✅  {service_name}: сохранено {len(final_gdf)} точек "
            f"({len(clean_points)} исходных + {len(polygons_as_points)} из полигонов)"
        )

## 4. Пакетный запуск

Автоматически обрабатывает все `.geojson` из `data/platform/after/`.  
Результаты сохраняются в `data/platform/` с исходными именами файлов.  
При ошибке алгоритм продолжает работу и фиксирует проблемный файл в итоговом отчёте.

In [10]:
INPUT_DIR = Path("data/platform/after")
OUTPUT_DIR = Path("data/platform")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Собираем список всех входных файлов
input_files = sorted(INPUT_DIR.glob("*.geojson"))
print(f"Найдено файлов для обработки: {len(input_files)}")
print()

results = []  # строки для итогового отчёта
failed = []   # имена файлов, завершившихся ошибкой

for input_path in input_files:
    output_path = OUTPUT_DIR / input_path.name
    try:
        msg = process_service_file(input_path, output_path, polygon_services)
        results.append(msg)
        print(msg)
    except Exception as e:
        err_msg = f"❌  {input_path.stem}: ОШИБКА — {e}"
        results.append(err_msg)
        failed.append(input_path.name)
        print(err_msg)

print()
print("=" * 60)
print(f"ИТОГ: обработано {len(input_files)} файлов")
print(f"  ✅  Успешно: {len(input_files) - len(failed)}")
print(f"  ❌  С ошибками: {len(failed)}")

if failed:
    print()
    print("Файлы с ошибками:")
    for f in failed:
        print(f"  - {f}")

Найдено файлов для обработки: 69

✅  animal_shelter: сохранено 4 точек (3 исходных + 1 из полигонов)
✅  bakery: сохранено 148 точек (142 исходных + 6 из полигонов)
✅  bank: сохранено 308 точек (306 исходных + 2 из полигонов)
✅  bar: сохранено 126 точек (125 исходных + 1 из полигонов)
✅  beauty: сохранено 225 точек (225 исходных + 0 из полигонов)
✅  books: сохранено 35 точек (34 исходных + 1 из полигонов)
✅  bus_station: сохранено 5 точек (2 исходных + 3 из полигонов)
✅  bus_stop: сохранено 1198 точек (1197 исходных + 1 из полигонов)
✅  cafe: сохранено 551 точек (529 исходных + 22 из полигонов)
✅  cemetery: сохранено 28 полигонов
✅  cinema: сохранено 20 точек (20 исходных + 0 из полигонов)
✅  clothes: сохранено 516 точек (513 исходных + 3 из полигонов)
✅  college: сохранено 53 точек (14 исходных + 39 из полигонов)
✅  convenience: сохранено 698 точек (671 исходных + 27 из полигонов)
✅  crematorium: сохранено 1 точек (0 исходных + 1 из полигонов)
✅  dentist: сохранено 165 точек (161 исход